In [1]:
import numpy as np

def compute_mae(var, var_hat):
    return np.sum(np.abs(var - var_hat)) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

In [2]:
import cvxpy as cp
import numpy as np

def robust_ar(y, d, p, delta, max_iter = 50):
    t = y.shape[0]
    vec = y[d :] # T-d
    mat = np.vstack([y[d-i-1 : t-i-1] for i in range(d)]).T # (T-d) x d
    theta = np.ones(t - d)
    for _ in range(max_iter):
        xt_w = mat.T * theta
        lhs = xt_w @ mat
        rhs = xt_w @ vec
        # w = np.linalg.pinv(lhs) @ rhs
        try:
            w = np.linalg.solve(lhs, rhs)
        except: #np.linalg.LinAlgError:
            w = np.linalg.lstsq(lhs, rhs, rcond = None)[0]
        epsilon = vec - mat @ w
        abs_eps = np.abs(epsilon)
        pos = np.where(abs_eps > delta)
        theta = np.ones(epsilon.shape)
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
    return w

def time_series_pred_multi(data, coef, horizon, step):
    t = data.shape[0]
    d = coef.shape[0]

    origins = np.arange(t - horizon, t) - step + 1             # (horizon,)
    assert origins.min() - d >= 0, "history too short"

    win = np.stack([data[o - d : o] for o in origins])         # horizon x d

    for _ in range(step):
        pred = win[:, ::-1] @ coef                             # (horizon,)
        win = np.hstack([win[:, 1:], pred[:, None]])

    return pred

(CVXPY) Oct 10 10:18:54 AM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Oct 10 10:18:54 AM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.11.4210). Expected < 9.10.0. Please open a feature request on cvxpy to enable support for this version.')


In [ ]:
import time
import pandas as pd
import numpy as np

data = pd.read_csv('../Wikipedia/top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values # .iloc[: no, 2 :].values

p = 1
delta = 20
step = 12
d = 7 * 24
horizon = 168
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : 24 * (31 - 14)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import time
import pandas as pd
import numpy as np

data = pd.read_csv('../Wikipedia/top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values # .iloc[: no, 2 :].values

p = 2/3
delta = 20
step = 12
d = 7 * 24
horizon = 168
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : 24 * (31 - 14)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import time
import pandas as pd
import numpy as np

data = pd.read_csv('../Wikipedia/top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values # .iloc[: no, 2 :].values

p = 1/2
delta = 20
step = 12
d = 7 * 24
horizon = 168
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : 24 * (31 - 14)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import time
import pandas as pd
import numpy as np

data = pd.read_csv('../Wikipedia/top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values # .iloc[: no, 2 :].values

p = 1/3
delta = 20
step = 12
d = 7 * 24
horizon = 168
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : 24 * (31 - 14)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import time
import pandas as pd
import numpy as np

data = pd.read_csv('../Wikipedia/top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values # .iloc[: no, 2 :].values

p = 0
delta = 20
step = 12
d = 7 * 24
horizon = 168
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : 24 * (31 - 14)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import numpy as np

data = np.load('../NA-temperature/daymet_tmax_na_2010.npz')['arr_0']
for t in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]:
    data = np.append(data, np.load('../NA-temperature/daymet_tmax_na_20{}.npz'.format(t))['arr_0'], axis = 1)

In [ ]:
import time
import numpy as np

p = 1
delta = 5
step = 1
d = 30
horizon = 365
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : - (365 + 365)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import time
import numpy as np

p = 2/3
delta = 5
step = 1
d = 30
horizon = 365
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : - (365 + 365)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import time
import numpy as np

p = 1/2
delta = 5
step = 1
d = 30
horizon = 365
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : - (365 + 365)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import time
import numpy as np

p = 1/3
delta = 5
step = 1
d = 30
horizon = 365
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : - (365 + 365)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()

In [ ]:
import time
import numpy as np

p = 0
delta = 5
step = 1
d = 30
horizon = 365
start = time.time()
est = np.zeros((data.shape[0], horizon))
for n in range(data.shape[0]):
    coef = robust_ar(data[n, : - (365 + 365)], d, p, delta)
    est[n, :] = time_series_pred_multi(data[n, :], coef, horizon, step)
end = time.time()
print('Running time: %d seconds'%(end - start))
ground = data[:, 24 * 31 - horizon : 24 * 31]
print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
print()